# 02 — Feature Extraction

**Goal**: Extract five types of NLP features from earnings call transcripts.

| Method | Session | Type |
|--------|---------|------|
| Loughran–McDonald Lexicon | 11 | Rule-based sentiment |
| N-grams | 3 | Bag-of-words frequency |
| TF-IDF | 7 | Weighted bag-of-words |
| Word2Vec | 13 | Dense word embeddings |
| FinBERT | 19-20 | Transformer sentiment |

In [1]:
import os
from pathlib import Path

# Move to project root so relative paths work (notebooks/ is one level down)
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))


## 1. Preprocessing

Before extracting features, we clean each transcript:
1. **Strip header**: remove participant lists, operator instructions, boilerplate
2. **Section split**: separate Presentation (exec remarks) from Q&A (analyst dialog)
3. **Normalise**: lowercase, remove punctuation

We do NOT remove stopwords for lexicon/FinBERT (need complete words).
For TF-IDF and Word2Vec, we remove stopwords to focus on content words.

In [2]:
from nasdaq_nlp.data.loader import scan_transcripts
from nasdaq_nlp.preprocessing.text import preprocess_transcript

records = scan_transcripts()
# Load and preprocess the first transcript as a demo
records[0].load()
result = preprocess_transcript(records[0].raw_text)

print("=== Presentation section (first 300 chars) ===")
print(result['presentation_raw'][:300])
print()
print("=== Q&A section (first 300 chars) ===")
print(result['qa_raw'][:300])
print()
print(f"Total tokens (full transcript): {len(result['tokens'])}")

=== Presentation section (first 300 chars) ===
presentation operator good day everyone and welcome to the apple incorporated second quarter fy earnings release conference call today's call is being recorded at this time for opening remarks and introductions i would like to turn the call over nancy paxton senior director of investor relations ple

=== Q&A section (first 300 chars) ===
questions and answers operator first we hear from simona jankowski with goldman sachs simona jankowski goldman sachs analyst my first question was actually just a clarification in terms of putting in context that billion in channel inventory reduction what was that last year just to help us make the

Total tokens (full transcript): 8910


## 2. Loughran–McDonald Lexicon Sentiment

**Math:**

$$\text{NegRate}_i = \frac{\#\text{negative words in transcript } i}{\text{total words}}$$
$$\text{PosRate}_i = \frac{\#\text{positive words in transcript } i}{\text{total words}}$$

These rates are our primary sentiment features. We normalise by total words
to control for transcript length (a 12,000-word call would naturally have
more negative words than an 8,000-word call, even at the same *rate*).

In [3]:
from nasdaq_nlp.features.lexicon import build_lexicon_features

lexicon_df = build_lexicon_features()
print(lexicon_df[['ticker','neg_rate','pos_rate','total_tokens']].describe().round(4))
print()
print("Top 5 most negative calls:")
print(lexicon_df.nlargest(5, 'neg_rate')[['ticker','neg_rate','pos_rate']].to_string())

Loaded LM dictionary: 347 positive, 2345 negative words
Computing lexicon features for 188 events...
Saved lexicon features → /Users/javierdominguezsegura/Academics/College/Courses/NLP/NASDAQ-NLP/outputs/processed/lexicon_features.csv  (188 rows)
  Avg NegRate: 0.0078
  Avg PosRate: 0.0143
       neg_rate  pos_rate  total_tokens
count  188.0000  188.0000      188.0000
mean     0.0078    0.0143     8214.2713
std      0.0020    0.0031     1219.2587
min      0.0045    0.0059     4502.0000
25%      0.0065    0.0122     7691.7500
50%      0.0074    0.0145     8389.5000
75%      0.0087    0.0166     8945.2500
max      0.0144    0.0219    12759.0000

Top 5 most negative calls:
   ticker  neg_rate  pos_rate
78   CSCO  0.014381  0.011676
91   CSCO  0.013749  0.016499
38   AMZN  0.013034  0.012900
82   CSCO  0.012925  0.014983
74   ASML  0.012533  0.011770


## 3. N-gram Features  (Session 3)

An n-gram is a contiguous sequence of n words.

- **Unigrams** (n=1): `['revenue', 'growth', 'exceeded']`
- **Bigrams** (n=2): `['revenue growth', 'growth exceeded']`

We count how often each n-gram appears in each transcript (bag-of-words).
The vocabulary is restricted to the top 500 n-grams by frequency.

Bigrams capture negation (`not strong`) and collocations (`market share`).

In [4]:
from pathlib import Path
from nasdaq_nlp.data.loader import scan_transcripts
from nasdaq_nlp.features.ngrams import build_ngram_matrix, get_top_ngrams

records = scan_transcripts()
file_paths = [r.file_path for r in records]

X_ng, features_ng, vec_ng = build_ngram_matrix(file_paths, ngram_range=(1,2), max_features=500)
print(f"N-gram matrix shape: {X_ng.shape}  (documents × features)")
print()
print("Top 20 n-grams by corpus frequency:")
print(get_top_ngrams(vec_ng, X_ng, top_n=20).to_string(index=False))

Preprocessing 188 transcripts for n-gram features...
N-gram matrix: 188 documents × 500 features
Vocabulary size: 500
N-gram matrix shape: (188, 500)  (documents × features)

Top 20 n-grams by corpus frequency:
    ngram  total_count  doc_frequency
     year        10014            188
    think         7126            188
  quarter         6728            188
    about         6698            188
   growth         5449            188
     just         5247            188
  revenue         5225            188
     time         5190            188
       up         4889            188
     it's         4793            188
     over         4760            188
     what         4549            188
    we're         4436            188
 business         4358            188
 question         4246            188
    there         4236            188
      all         3976            188
       us         3770            188
      see         3746            188
customers         3725       

## 4. TF-IDF  (Session 7)

TF-IDF weights each word by how important it is to a specific document,
relative to the whole corpus.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t, d)}{|d|}}_{\text{TF}} \times \underbrace{\log\frac{N}{df(t)}}_{\text{IDF}}$$

- **TF**: how often term $t$ appears in document $d$
- **IDF**: inverse document frequency — high if $t$ is rare across all $N$ documents

A high TF-IDF score means the term is frequent in *this* document but rare elsewhere
→ it characterises this document specifically.

In [5]:
from nasdaq_nlp.features.tfidf import build_tfidf_features, top_tfidf_terms_by_ticker
import pandas as pd

tfidf_df, vectorizer = build_tfidf_features()
print(f"TF-IDF feature matrix: {tfidf_df.shape}")
print()

events = pd.read_csv('outputs/processed/event_study_dataset.csv')
X_tfidf = tfidf_df.filter(like='tfidf_').values
top_terms = top_tfidf_terms_by_ticker(events, vectorizer, X_tfidf, top_n=5)
print("Top 5 characteristic terms by ticker:")
print(top_terms.to_string(index=False))

Building TF-IDF matrix for 188 transcripts...
TF-IDF matrix: 188 docs × 500 features
Saved TF-IDF features → /Users/javierdominguezsegura/Academics/College/Courses/NLP/NASDAQ-NLP/outputs/processed/tfidf_features.csv  ((188, 503))
TF-IDF feature matrix: (188, 503)

Top 5 characteristic terms by ticker:
ticker              term  mean_tfidf  rank
  AAPL            iphone    0.212826     1
  AAPL           sync id    0.204024     2
  AAPL                id    0.198372     3
  AAPL             apple    0.190423     4
  AAPL              sync    0.185966     5
   AMD              lisa    0.200171     1
   AMD           lisa su    0.185243     2
   AMD                su    0.185243     3
   AMD     micro devices    0.178379     4
   AMD    advanced micro    0.178379     5
  AMZN             prime    0.228653     1
  AMZN            amazon    0.183662     2
  AMZN             brian    0.158625     3
  AMZN               com    0.105106     4
  AMZN               inc    0.083848     5
  ASML   

In [6]:
tfidf_df

,ticker,file_name,event_trading_day,tfidf_able,tfidf_about,tfidf_across,tfidf_actually,tfidf_add,tfidf_advanced,tfidf_advanced micro,...,tfidf_work,tfidf_working,tfidf_world,tfidf_year,tfidf_year over,tfidf_year year,tfidf_years,tfidf_you're,tfidf_you've,tfidf_youtube
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-27,0.041487,0.064543,0.014681,0.025255,0.000000,0.000000,0.0,...,0.000000,0.000000,0.044484,0.059055,0.031805,0.000000,0.030004,0.039286,0.024725,0.000000
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-27,0.017858,0.078731,0.046037,0.035161,0.017954,0.038275,0.0,...,0.034976,0.044186,0.053459,0.078731,0.044422,0.022866,0.055966,0.037719,0.034424,0.000000
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-27,0.040031,0.079961,0.056315,0.037557,0.019177,0.000000,0.0,...,0.046451,0.030623,0.057100,0.069452,0.043390,0.000000,0.044761,0.040289,0.045719,0.000000
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-26,0.038963,0.078203,0.050504,0.017418,0.039171,0.023501,0.0,...,0.051042,0.042009,0.061304,0.080617,0.049408,0.049887,0.060757,0.060934,0.047608,0.000000
4,AAPL,2017-Jan-31-AAPL.txt,2017-02-01,0.033730,0.068932,0.025129,0.044422,0.000000,0.034447,0.0,...,0.025396,0.015240,0.053071,0.065994,0.042773,0.000000,0.044509,0.033947,0.024996,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,NVDA,2019-Aug-15-NVDA.txt,2019-08-16,0.057316,0.069984,0.046214,0.040135,0.000000,0.000000,0.0,...,0.043656,0.035673,0.061458,0.071899,0.028934,0.067618,0.042067,0.048862,0.027880,0.048356
184,NVDA,2019-Nov-14-NVDA.txt,2019-11-15,0.057007,0.070587,0.039288,0.016728,0.017926,0.038214,0.0,...,0.053201,0.049805,0.073285,0.076470,0.056132,0.059572,0.047235,0.048598,0.027729,0.000000
185,NVDA,2020-Feb-13-NVDA.txt,2020-02-14,0.055763,0.073112,0.042025,0.000000,0.000000,0.000000,0.0,...,0.042472,0.048718,0.066669,0.072750,0.016625,0.065785,0.050146,0.047538,0.016020,0.000000
186,NVDA,2020-May-21-NVDA.txt,2020-05-22,0.040395,0.064893,0.039907,0.030479,0.026351,0.041123,0.0,...,0.046190,0.055376,0.068645,0.061285,0.014757,0.058390,0.049628,0.047750,0.014219,0.041757


## 5. Word2Vec Document Embeddings  

Word2Vec learns a dense vector representation for each word by training a
neural network to predict surrounding words (skip-gram architecture).

**Key property**: semantically similar words cluster together:
$$\text{cosine}(\vec{\text{strong}}, \vec{\text{robust}}) \approx 1$$

To represent an entire transcript (document), we **average-pool** word vectors:
$$\vec{d} = \frac{1}{|tokens|} \sum_{w \in d} \vec{w}$$

Each document is now a 100-dimensional dense vector, capturing overall semantic content.

In [7]:
from nasdaq_nlp.features.embeddings import build_embedding_features, nearest_neighbors

emb_df, w2v_model = build_embedding_features()
print(f"Embedding matrix: {emb_df.shape}")
print()
# Sanity check: nearest neighbors for financial terms
for word in ['growth', 'risk', 'revenue', 'strong', 'guidance']:
    neighbors = nearest_neighbors(w2v_model, word, top_n=5)
    if neighbors:
        nn_str = ', '.join(f"{w}({s:.2f})" for w, s in neighbors)
        print(f"  '{word}' → {nn_str}")

Tokenising 188 transcripts for Word2Vec...
Corpus: 188 documents, 910,973 total tokens
Training Word2Vec: vector_size=100, window=5, min_count=2, epochs=10


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec vocabulary: 10090 words
Saved Word2Vec model → /Users/javierdominguezsegura/Academics/College/Courses/NLP/NASDAQ-NLP/outputs/processed/word2vec.model
Computing document embeddings (average pooling)...
Saved embeddings → /Users/javierdominguezsegura/Academics/College/Courses/NLP/NASDAQ-NLP/outputs/processed/doc_embeddings.csv  ((188, 103))
  Nearest to 'growth': exhibit(0.71), grower(0.69), performer(0.68), oversupplied(0.67), upsell(0.67)
  Nearest to 'risk': tolerance(0.70), resolved(0.67), obsolescence(0.65), considerations(0.62), geopolitical(0.58)
  Nearest to 'revenue': revenues(0.76), sites'(0.75), outperformed(0.74), site's(0.72), mbu(0.71)
  Nearest to 'strong': solid(0.84), healthy(0.72), strength(0.70), admob(0.67), fueled(0.67)
Embedding matrix: (188, 103)

  'growth' → exhibit(0.71), grower(0.69), performer(0.68), oversupplied(0.67), upsell(0.67)
  'risk' → tolerance(0.70), resolved(0.67), obsolescence(0.65), considerations(0.62), geopolitical(0.58)
  'revenue' → r

## 6. FinBERT Sentiment  (Sessions 19-20)

FinBERT is BERT fine-tuned on financial text. Unlike the lexicon, it understands
context (negation, idioms, domain jargon).

For each sentence in the transcript:
$$P(\text{positive}), P(\text{negative}), P(\text{neutral}) \quad \text{(sum to 1)}$$

We average across all sentences in the transcript:
$$\bar{P}(\text{negative}) = \frac{1}{S}\sum_{s=1}^{S} P_s(\text{negative})$$

In [8]:
from pathlib import Path
finbert_path = Path('outputs/processed/finbert_features.csv')

if finbert_path.exists():
    import pandas as pd
    fb = pd.read_csv(finbert_path)
    print(f"FinBERT features: {len(fb)} events")
    print(fb[['ticker','finbert_pos_mean','finbert_neg_mean','finbert_neu_mean']].describe().round(3))
else:
    print("FinBERT features not yet computed.")
    print("Run: from nasdaq_nlp.features.finbert import build_finbert_features; build_finbert_features()")
    print("(Takes ~20-30 min on CPU)")

FinBERT features not yet computed.
Run: from nasdaq_nlp.features.finbert import build_finbert_features; build_finbert_features()
(Takes ~20-30 min on CPU)


In [ ]:
# Verification: feature extraction complete
import numpy as np
assert tfidf_df.shape == (188, 503), f"Unexpected TF-IDF shape: {tfidf_df.shape}"
assert emb_df.shape == (188, 103), f"Unexpected embedding shape: {emb_df.shape}"
print("✓ All feature extraction steps verified")
print(f"  Lexicon:    {len(lexicon_df)} events × 5 features")
print(f"  TF-IDF:     {tfidf_df.shape[0]} events × {tfidf_df.shape[1]-3} features")
print(f"  Word2Vec:   {emb_df.shape[0]} events × {emb_df.shape[1]-3} dimensions")